In [21]:
import sys
from pathlib import Path

# Add project root to Python path
ROOT_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT_DIR))


import json
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from src.ingest import load_machine_data, build_index

GROUND_TRUTH_OUTPUT_PATH = ROOT_DIR / "data"
RAG_ANSWER_PATH = ROOT_DIR / "data"
RAG_EVALUATE_PATH = ROOT_DIR / "data"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
PROJECT_ROOT = Path.cwd().parent
GROUND_TRUTH_OUTPUT_PATH = ROOT_DIR / "data"
GROUND_TRUTH_OUTPUT_PATH 

WindowsPath('D:/DataTalksClub/Submission/Predictive-Maintenance-RAG-Assistant/data')

# 1. Search Evaluation

In [23]:
# load ground_truth file
loaded_ground_truth = pd.read_csv(GROUND_TRUTH_OUTPUT_PATH / "ground_truth.csv")
loaded_ground_truth.head(3)

,question,query_section,document
0,What signs usually show up before tool wear fa...,Tool Wear Failure,299b2d2994
1,At about how many minutes of continuous use do...,Tool Wear Failure,299b2d2994
2,Can a tool wear failure happen without any sen...,Tool Wear Failure,299b2d2994


In [24]:

loaded_ground_truth_dict = loaded_ground_truth.to_dict(orient='records')
q= loaded_ground_truth_dict[0]
q

{'question': 'What signs usually show up before tool wear failure, or can it just fail even when torque, temperature, and speed all look normal?',
 'query_section': 'Tool Wear Failure',
 'document': '299b2d2994'}

# 2. Prepare evaluation functions

In [25]:
# prepare required functions
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)


In [26]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }





# 3. Prepare search text

In [27]:
# load machine data
documents = load_machine_data()

# Set to a specific section name (e.g. 'Power Failure') to restrict retrieval
# evaluation to that section only; set to None (default) to evaluate against
# the whole knowledge base, matching the full-coverage ground_truth.csv.
# NOTE: if you do scope this down, filter loaded_ground_truth_dict to the same
# section before calling evaluate(), otherwise hit_rate/mrr will be measured
# against questions whose target document was never indexed at all.
SECTION_FILTER = None

documents_eval = [
    doc for doc in documents
    if SECTION_FILTER is None or doc["section"] == SECTION_FILTER
]
len(documents_eval)


Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71


71

# 4. Using lexical text search

In [28]:
index_eval = build_index(documents_eval)

def text_search(query):
    boost_dict = {
        "question": 2.0,
        "section": 0.5
    }

    return index_eval.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

text_metrics = evaluate(loaded_ground_truth_dict, text_search)
print("Text search:", text_metrics)

  0%|          | 0/355 [00:00<?, ?it/s]

Text search: {'hit_rate': 0.6816901408450704, 'mrr': 0.48535211267605644}


# 5. Using cosine similarity

In [29]:
# Fields used for retrieval
document_texts = [
    " ".join([
        doc.get("process_control") or "",
        doc.get("failure_mode") or "",
        doc.get("section") or "",
        doc.get("question") or "",
        doc.get("answer") or ""
    ])
    for doc in documents_eval
]
vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(document_texts)

In [30]:
def vector_search(query, top_k=5):
    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        document_vectors
    )[0]

    top_indexes = scores.argsort()[::-1][:top_k]

    results = []

    for index in top_indexes:
        results.append({
            **documents_eval[index],
            "score": round(float(scores[index]), 4)
        })

    return results


In [31]:
vector_metrics = evaluate(loaded_ground_truth_dict, vector_search)

print("Vector search:", vector_metrics)

  0%|          | 0/355 [00:00<?, ?it/s]

Vector search: {'hit_rate': 0.8788732394366198, 'mrr': 0.7001408450704224}


In [32]:
print("Text search:", text_metrics)
print("Vector search:", vector_metrics)

Text search: {'hit_rate': 0.6816901408450704, 'mrr': 0.48535211267605644}
Vector search: {'hit_rate': 0.8788732394366198, 'mrr': 0.7001408450704224}


It is obvious that vector search gives us better results when evaluating retrieval performance.

We will use this search method in our RAG pipeline going forward, since it more consistently retrieves the correct source document across the different failure-mode sections of the knowledge base.